## **RAG 구조를 구현하여 외부 지식을 참조하는 대화형 AI 구현**

In [59]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# langchain과 openai가 연동된 pre-trained Embedding모델 사용
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

### 1. gpt-3.5-turbo 모델에게 일반적 질의

In [61]:
load_dotenv()
MY_API_KEY = os.getenv('OPENAI_API_KEY')

In [62]:
chat_model = ChatOpenAI(
    model='gpt-3.5-turbo',
    api_key=MY_API_KEY,
    temperature=0.1
)

chat_model.invoke('기술평가가 뭐야?').content

'기술평가는 특정 기술이나 제품의 성능, 품질, 안전성, 신뢰성 등을 평가하는 과정을 말합니다. 이를 통해 해당 기술이나 제품의 우수성과 개선이 필요한 부분을 파악할 수 있습니다. 주로 실험, 시험, 분석 등의 방법을 사용하여 기술적인 측면을 평가하며, 이를 통해 보다 나은 제품이나 서비스를 개발할 수 있습니다.'

### 2. RAG 구조와 벡터 DB를 활용한 질의
#### 1) 참조할 데이터 로드 및 청킹

In [63]:
# 한번에 여러개의 PDF 로드
loaders = [PyPDFLoader('data/기술보증기금과 한국경제.pdf'),  # 5page
           PyPDFLoader('data/기술보증기금 주요업무.pdf')  # 4page
           ]

In [64]:
# Recursive Splitter 객체 생성
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,               # 사내 기술 문서이므로 청크 사이즈를 상대적으로 크게
    chunk_overlap=50,
    separators=['\n\n', '\n', ' ']
)

In [65]:
loaders  # 리스트 내부에 2개의 pdf가 있음

In [66]:
my_chunks = []  # 청크들을 보관할 빈 리스트

for loader in loaders :
    # 2개의 PDF 파일에서 추출된 document 객체들을 하나의 리스트로 합쳐줌
    # 기존 리스트에서 또 다른 리스트(my_chunks)로 값만 옮기기 위해 extend 사용
    my_chunks.extend(loader.load_and_split(pdf_splitter))

print(len(my_chunks))
print()
my_chunks

# 두 파일은 총 9페이지였지만, chunk_size=1000으로 해서 청킹한 결과 13개

13



[Document(metadata={'producer': 'Microsoft® Word Office 365용', 'creator': 'Microsoft® Word Office 365용', 'creationdate': '2019-06-27T22:46:56+09:00', 'author': 'HS', 'moddate': '2019-06-27T22:46:56+09:00', 'source': 'data/기술보증기금과 한국경제.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='페이지 1 / 5 \n \n기술보증기금과 한국경제 \n \nI. 기술보증기금 개요  \n1. 설립근거 : 기술보증기금법 \n \n- 설립목적(존재이유) \n✓ 기술보증기금을 설립하여 기술보증제도를 정착·발전시킴으로써 신 기술사업에 대한 자금의 \n공급을 원활하게 하고 나아가 국민 경제의 발전에 이바지함을 목적으로 함(기술보증기금법1\n조) \n✓ 설립 : 담보능력이 미약한 기업의 채무를 보증하게 하여 기업에 대한 자금 융통을 원활하게 하기 \n위하여 기술보증기금을 설립(법 12조) \n✓ 기금의 재원 : 정부 출연금의 예산은 중소벤처기업부 소관으로 함 \n \n☞ 기술보증기금은  \n✓ 기술력은 우수하지만 담보 부족한 중소기업의 \n✓ 기술성과 사업성 평가를 통해 기술보증을 지원하며, \n✓ 기술평가, 벤처이노비즈기업  인증, 중소기업 창업지원 등의 업무 수행 \n \n2. 주요개념1 \n \n업  무 내  용 \n기술보증 신기술사업자가 부담하는 금전 채무 보증.( 신용부족-담보부족 해결) \n \n금융회사 등으로부터 자금 대출을 받음으로써 금융회사 등에 대하여 부담하는 금전 채무를 \n기술보증기금이 기술보증서로 보증 \n 신기술사업자 - 기술을 개발하거나 이를 응용하여 사업화하는 중소기업(「중소기업기본법」에 \n따른 중소기업) 및 대통령령으로 정하는 기업 \n- "기업"이란 사업을 하는 개인 및 법인 \n신용보증 상시 사용하

#### 2) 임베딩 설정
- Sentence Transformers (내 컴퓨터 CPU, GPU 사용)
- OpenAI Embedding (금액이 드는 대신 OepnAI 컴퓨터가 대신함)
- oepnai에 임베딩 모델, 설명: https://developers.openai.com/api/docs/guides/embeddings

In [67]:
# 임베딩 객체 생성
 # text-embedding-3-small : 모델의 최대 입력가능 토큰 수는 8191개, 벡터의 차원은 1536차원
 # text-embedding-3-large : 최대 입력가능 토큰 수 8191개, 벡터 차원은 3072차원
my_embedding = OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=MY_API_KEY
)

In [68]:
# 임베딩 예시
my_list = ['이제 봄이 오려나봐요!', '아직은 좀 춥긴 하지만..ㅠ']

# embed_documents : 임베딩 진행 함수
temp = my_embedding.embed_documents(my_list)
print('my_list 길이 :', len(temp))
print('my_list 내 첫 문장 길이: ', len(temp[0]))
print(temp)

# 두 개의 문자열이 각각 1536차원으로 임베딩 된 것을 확인

my_list 길이 : 2
my_list 내 첫 문장 길이:  1536
[[0.0248260498046875, -0.00983428955078125, -0.04296875, -0.03948974609375, -0.0113525390625, -0.01351165771484375, -0.0019083023071289062, -0.00786590576171875, 0.0241241455078125, -0.06866455078125, -0.00921630859375, 0.0179443359375, -0.01471710205078125, 0.0016965866088867188, -0.008544921875, -0.0185089111328125, -0.059326171875, -0.005069732666015625, -0.0097198486328125, -0.00397491455078125, 0.0230712890625, -0.01517486572265625, -0.01392364501953125, -0.0216827392578125, 0.0252227783203125, 0.03936767578125, 0.02740478515625, 0.012359619140625, 0.08026123046875, 0.00244903564453125, -0.0026092529296875, -0.0281524658203125, 0.0206146240234375, -0.01100921630859375, 0.016937255859375, -0.027008056640625, -0.0233306884765625, -7.075071334838867e-05, -0.0113525390625, -0.059600830078125, 0.01215362548828125, 0.00957489013671875, -0.00467681884765625, 0.01177978515625, 0.034881591796875, 0.042144775390625, 0.005771636962890625, 0.00480270385

#### 3) 벡터 DB 생성
- chorma DB 공식 홈페이지: https://www.trychroma.com/

In [69]:
!pip install langchain-chroma

In [70]:
# langchain 지원 Chroma 벡터 DB 클래스 임포트
from langchain_chroma import Chroma

# Chroma DB의 데이터 저장 방식은 기본적으로 로컬 저장 (디스크 저장방식)

In [81]:
# 데이터 저장 경로
my_directory = '/VectorStores'

                # from_documents : 각종 정보로 DB를 생성하는 함수
vectordb = Chroma.from_documents(
    documents=my_chunks,                  # 벡터화 시킬 청크 목록
    embedding=my_embedding,               # 임베딩 모델
    persist_directory=my_directory,       # 데이터 저장방식
    # persist_directory 사용하면 디스크 저장 방식으로 로컬에 저장, 없으면 메모리 저장
    collection_metadata={'hnsw:space':'cosine'}    # 유사도 계산 방식
)

#### <생성된 DB 불러와서 사용하기 (껐다 켰을 경우)>
- 같은 경로에 DB를 생성하면 데이터가 누적저장됨
- 한 번 생성된 DB가 있다면 새롭게 객체를 생성하지 말고, 기존 DB를 불러와서 사용해야 함

In [82]:
my_directory = '/VectorStores'

vectordb = Chroma(
    persist_directory=my_directory,
    # 기존 DB를 불러올 때는 embedding이 아니라, embedding_function으로 설정
    embedding_function=my_embedding
)

>#### Chroma객체(벡터DB)에서 자주 사용하는 메소드
- _collection.count : 수집된 벡터의 개수 확인
- _collection.get : 수집된 벡터의 정보 확인
- delete : 특정 id를 가진 벡터 삭제
- delete_collection : 모든 데이터 삭제
- add_documents : 여러 문서를 벡터 저장소에 추가
- similarity_search : 유사도 계산을 통해 입력 질의에 대해 가장 유사한 문서 검색

#### 1) DB 정보 확인

In [83]:
# 수집된 청크 벡터 개수 확인 (값이 없으면 에러 발생)
vectordb._collection.count()

# 벡터 DB 생성 코드를 여러번 실행하면 값이 누적 저장됨~!

13

In [74]:
# 수집된 벡터들의 id, 원본 문서 텍스트, 문서위치 등의 정보를 출력
vectordb._collection.get()

{'ids': ['9cc8fbe5-4776-4952-9d5c-fb9b98ce2573',
  '8a917b06-1f5d-45b0-8a9f-93c1d49a4a6d',
  '04cd492b-a7fa-44b5-b3c3-68005dfbf4c5',
  'b39f3ba8-802c-4982-81d6-84130842a44b',
  'ab761ef0-3e24-4ece-9b95-cb73c70fb02c',
  '0752787b-155e-4a5f-9a8f-2fcdd7b11cf4',
  '2febd49a-fe35-447f-aef8-cec478b90fd7',
  'c72a417b-0de2-48e7-82a3-d8c395a68ffc',
  '3a5b2418-79a7-4e28-8755-996342250b99',
  '84c783d5-812d-460e-8b6a-cef510099072',
  'bfba3c52-c390-41c9-9247-d59834558c29',
  'd03d1ae4-1b36-484e-8fb8-9d9277ab4efd',
  'd4825b1b-a89d-4d0c-8b6a-06b82ff3b288',
  '4fc4a6ab-ec1f-4261-adb3-ea5392f2d3cb',
  'd1ed4509-2db6-409d-8120-663031d77ead',
  'ff1e4fa9-6517-451d-a2d3-fc38005100a3',
  'a6fc341d-d8cc-4ac7-b238-c3a073710aa1',
  '0bfc8178-211e-472b-b10e-275b24f7236b',
  '8c5bb588-7d90-4522-8527-c47a7feb4d08',
  '2b03817f-c1b4-463b-9363-da4855f8b525',
  'c0091580-ec67-4a7f-8dae-d5cfbc1501ed',
  'd6aa1d5c-e75a-4f7d-90a2-9d7462361abe',
  'c420574e-a6eb-40f4-a802-29416d86d371',
  '185fd73f-a401-43c5-a1ac-

#### 2) 기존 DB에 문서 추가하기

In [101]:
add_loaders = [PyPDFLoader('data/기술보증기금 주요업무_기술평가.pdf'),
               PyPDFLoader('data/기술보증기금 주요업무_기술보증.pdf')
               ]

add_chunks= []
for loader in add_loaders :
    add_chunks.extend(loader.load_and_split(pdf_splitter))

print('새로 추가할 문서 청크 수:', len(add_chunks))

새로 추가할 문서 청크 수: 15


In [103]:
# 벡터 DB에 문서 추가 (추가 후, 추가된 각 벡터에 대한 id값이 확인차 출력됨)
vectordb.add_documents(add_chunks)

['f8d01d6e-969f-469a-adbd-05a4213840b1',
 'c0ce2b1a-48fd-4e09-9eb5-de7e0441f858',
 '38060412-d1b7-4f5e-a407-5f6c4d197e70',
 '89501f51-5ffe-47a5-b081-5d85e56ef74b',
 '7e67f22e-e484-4646-94fd-35033ab0e024',
 '5f7d3e21-0bf8-4094-a63f-44476b772912',
 '6eb379b2-fbeb-4796-a8ca-c0f2ef535dbb',
 'ac4c7e9f-f47b-4e14-954b-a16752d7af26',
 '23e4a6d4-c2e9-4fc4-a1ba-8530ac9aacd0',
 'ffef3f2e-7044-4597-8f75-7d1e96990dfd',
 '4ae2999c-7e51-463f-8294-653a7598e7b7',
 '3b20f750-3eac-4a7c-a01e-e2c8c48b0e9f',
 '1b133067-6cde-4bf2-bddb-1fcdca144c94',
 '61d2ebcc-5e2a-4033-9284-d0cdcffb5084',
 'a7df7d54-8ecc-4e13-8e12-dbdb2ec2e82c']

In [104]:
vectordb._collection.count()

# 기존 DB의 임베딩 벡터 13 + 추가한 15 = 28

28

#### 3) DB 값 삭제

In [ ]:
# # 벡터의 id 값으로 삭제
# vectordb.delete(ids=['27ff7acd-edb5-41df-be00-dbc34bda109f'])
# vectordb._collection.count()

56

In [ ]:
# # 모든 데이터 삭제
# vectordb.delete_collection()
# vectordb._collection.count()  # 값이 없으면 에러 발생

ValueError: Chroma collection not initialized. Use `reset_collection` to re-create and initialize the collection. 

#### 4) 유사도 검색
- 가장 가까운 것 찾기. 정확도는 높지만 중복될 수 있음

In [ ]:
question = "기술평가가 뭐야?"

# smilarity_search : 유사도 검색 실행
 # k : 반환 받고자 하는 유사도가 높은 벡터(document 객체) 수 지정
 # 값이 너무 크면 연관없는 벡터도 검색될 수 있음 (ex 벡터 DB에 벡터가 10개인데 10으로 설정하면 모두 반환됨)
similar_docs = vectordb.similarity_search(question, k=3)

print('객체 수: ', len(similar_docs))

In [ ]:
print(similar_docs[0].page_content)

#### 5) MMR(Max Marginal Relevance) 유사도 검색
- 가장 주변부 유사도
- 사용자 질의에 가장 관련성이 높은 벡터를 선택하지만, 중복된 정보를 줄이고 `다양한 정보`를 제공하는 것을 목표로 하는 검색 방법

In [ ]:
search_mmr = vectordb.max_marginal_relevance_search(
    question,
    fetch_k=6,           # 벡터 DB에서 가져올 유사도가 높은 문서 수
    k=3,                 # '응답의 다양성'을 고려해 fetch_k 내에서 반환할 문서 수
    lambda_mult=0.5     # 0~1 사이 실수값으로 관련성과 다양성 사이의 균형을 제어
                        # (0에 가까울수록 다양성 우선, 1에 가까울 수록 관련성 우선)
)

print(search_mmr[0].page_content)
# 즉, 질의에 대해 유사도가 높은 6개의 문서 선택, 그 중 다양한 3개를 추출하는 방식

- 일반 유사도 검색시 page_label 2,5,4인데 mmr은 2,6,5 나옴

#### 6) as_retriever 클래스 사용
- as_retriever는 벡터 DB객체를 **효율적인 검색기**로 사용할 수 있도록 변환해주는 역할
- as_retriever를 사용하면 langchain 생태계와의 연동성이 좋아지고, 질의를 더 정교하게 인지하고 처리하며,
- 유사도 검색 전후에 필터나 제약조건 등을 추가로 적용할 수 있음. 성능이 더 좋아서 langchain에서 LLM과 연결시 주로 사용됨

In [ ]:
my_retriever = vectordb.as_retriever(
    search_type='similarity',    # 일반 유사도 검색
    search_kwargs={'k':3}
)

# invoke: 검색기에 질의 입력하여 유사도가 높은 문서(벡터) 반환
my_retriever.invoke(question)

In [116]:
# 유사도 조건 검색
# score_threshold 문턱~!
# (모르면 모른다고 대답해야 할 경우는 커트라인이 필요 ex. 사내 규정 챗봇, 고객센터QA 챗봇)

my_retriever2 = vectordb.as_retriever(
    search_type='similarity_score_threshold',    # 유사도 점수 커트라인 적용
    search_kwargs={
        'k':3,                  # 최대 3개까지 찾되
        'score_threshold':0.5   # 유사도 점수가 50점 이상인 문서만 찾기
        }
)
# (점수의 기준은 embedding 모델과 벡터DB의 종류마다 달라서, 프로토 타입으로 최상위 문서들의 점수를 먼저 확인해보고
# 질의 내용과 달라지는 문서의 점수 이상으로 설정하는 것이 좋음)

my_retriever2.invoke(question)

[Document(id='c0ce2b1a-48fd-4e09-9eb5-de7e0441f858', metadata={'producer': 'Microsoft® Word Office 365용', 'author': 'HS', 'page': 1, 'source': 'data/기술보증기금 주요업무_기술평가.pdf', 'creator': 'Microsoft® Word Office 365용', 'total_pages': 6, 'page_label': '2', 'moddate': '2019-06-28T00:58:11+09:00', 'creationdate': '2019-06-28T00:58:11+09:00'}, page_content='페이지 2 / 6 \n \n (2) 기술평가3 \n기술평가란?- 대상기술의 기술성, 시장성, 사업타당성 등을 분석하고 결과를 금액, 등급, 의견 등으로 \n표현하는 것이다.  \n✓ 기술평가는 기술수준 등 기술자체에 대한 평가를 표현한 용어지만, 기술과 기업(사업)간의 밀접한 관련\n성으로 인해서 최근에는 위의 의미로 사용됨 \n✓ 기술가치평가는 기술평가+가치평가가 결합된 용어로서, 기술적인 요소를 기반으로 시장에서의 가치를 \n평가 하는 것을 의미 \n \n (3) 기술평가 유형 \n 정 의 세부평가종류 \n기술가치평\n가 \n대상 기술에 의하여 현재 시현되고 있거나 \n장래에 시현될 기술의 가치를 평가하여 평\n가결과를 금액으로 표시 \n- 벤처기업 현물출자 특례대상 산업재산권  등의 평가 \n- 기술의 담보가치를 산정하기 위한 평가 \n- 기술이전ㆍ거래4 기준가격 산정을 위한   평가 \n- 기술사업의 이전ㆍ양수도를 위한 평가 등  \n기술사업 \n타당성평가 \n기업이 특정기술(OR 아이디어)을 사업화, \n추진중인 기술사업의 투자 확대 시 사업의 \n기술성 및 사업타당성 평가 \n- 벤처기업 확인, INNO-BIZ 선정평가 \n- 정책자금 지원 대상자 선정 위한 평가 \n- 금융기관 등의 여신심사용 기술평가 \n- 보증지원을 위한 평가 

In [117]:
# 벡터와 질의 간 유사도 점수 출력
results = vectordb.similarity_search_with_relevance_scores(
    '기술평가가 뭐야?',
    k=3
)

print(len(results))
print()
results

# 유사도가 높은 상위 3개의 문서에 대한 유사도 점수 출력

3



[(Document(id='c0ce2b1a-48fd-4e09-9eb5-de7e0441f858', metadata={'producer': 'Microsoft® Word Office 365용', 'page': 1, 'creationdate': '2019-06-28T00:58:11+09:00', 'source': 'data/기술보증기금 주요업무_기술평가.pdf', 'total_pages': 6, 'creator': 'Microsoft® Word Office 365용', 'author': 'HS', 'moddate': '2019-06-28T00:58:11+09:00', 'page_label': '2'}, page_content='페이지 2 / 6 \n \n (2) 기술평가3 \n기술평가란?- 대상기술의 기술성, 시장성, 사업타당성 등을 분석하고 결과를 금액, 등급, 의견 등으로 \n표현하는 것이다.  \n✓ 기술평가는 기술수준 등 기술자체에 대한 평가를 표현한 용어지만, 기술과 기업(사업)간의 밀접한 관련\n성으로 인해서 최근에는 위의 의미로 사용됨 \n✓ 기술가치평가는 기술평가+가치평가가 결합된 용어로서, 기술적인 요소를 기반으로 시장에서의 가치를 \n평가 하는 것을 의미 \n \n (3) 기술평가 유형 \n 정 의 세부평가종류 \n기술가치평\n가 \n대상 기술에 의하여 현재 시현되고 있거나 \n장래에 시현될 기술의 가치를 평가하여 평\n가결과를 금액으로 표시 \n- 벤처기업 현물출자 특례대상 산업재산권  등의 평가 \n- 기술의 담보가치를 산정하기 위한 평가 \n- 기술이전ㆍ거래4 기준가격 산정을 위한   평가 \n- 기술사업의 이전ㆍ양수도를 위한 평가 등  \n기술사업 \n타당성평가 \n기업이 특정기술(OR 아이디어)을 사업화, \n추진중인 기술사업의 투자 확대 시 사업의 \n기술성 및 사업타당성 평가 \n- 벤처기업 확인, INNO-BIZ 선정평가 \n- 정책자금 지원 대상자 선정 위한 평가 \n- 금융기관 등의 여신심사용 기술평가 \n- 보증지원을 위한 평가

In [118]:
# 내용(doc) 뒤에 점수(score) 나오므로 각각 받기
for idx, (doc, score) in enumerate(results) :
    print(f'[순위 {idx+1}] 점수: {score:.4f}')    # 순위와 점수 출력
    print(f'내용: {doc.page_content[:40]}...\n') # 내용은 중략해서 출력

[순위 1] 점수: 0.5734
내용: 페이지 2 / 6 
 
 (2) 기술평가3 
기술평가란?- 대상기술의 기...

[순위 2] 점수: 0.5624
내용: 페이지 5 / 6 
 
및 제시 
 
③ 기타 가치평가 
기타 기술가치평...

[순위 3] 점수: 0.5473
내용: 페이지 4 / 6 
 
2. 기술평가의 활용방안5 
활용방안 주요내용 
...



- 표준 유사도 검색기는 유사도가 높은 문서 상위 k개를 빠르게 검색하므로, 관련성이 높은 문서를 빠른 속도로 검색하기 좋고
- mmr 유사도 검색기는 검색의 중복을 줄이고, 다양성을 포함해야 할 필요가 있을 때 사용 (추천과 같이 원하는 답변이 여러 종류인 경우, 대신 표준에 비해 속도는 느림)

#### 7) LCEL Chain 구성
- **LCEL(LangChain Expression Language)** : AI 작업 전반을 빠르게 처리하고 투명하게 관리할 수 있는 langchain 표현 언어
- 파이프(|) 기호를 사용해 앞 단계의 결과물(output)을 다음 단계의 입력(input)으로 전달하는 직관적 구조를 가짐
- Langchain의 모든 객체들이 Runnable(실행가능한) 객체로 구성되어 있으며 `프롬프트, LLM, 검색기` 모두 규격에 맞춰 만들어져서 chain으로 연결할 수 있음
- 전체 프로세스를 투명하게 관리하기 때문에 LangSmith와 연동하여 시각적으로 내부 추적 가능

>#### chain = prompt | model | output_parser

In [ ]:
# Chain 내부에서 동작할 데이터 정의와 검증을 위한 모듈
!pip install -q pydantic

In [94]:
# 단순 문자열이 아닌 메시지 리스트 형태로 역할(role)을 지정한 프롬프트 템플릿
# (단순 문자열인 PromptTemplate에 비해 role이 지정되어 더 높은 퀄리티의 프롬프트 작성)
from langchain_core.prompts import ChatPromptTemplate

# 입력으로 들어온 데이터를 다음 단계로 넘겨주는 클래스
# (langchain 생태계에서 내부 로직 파악하고 시각화하기 위해 지정된 형식)
from langchain_core.runnables import RunnablePassthrough

# LLM이 응답한 복잡한 결과물에서 순수 텍스트만 추출해주는 클래스
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# 검색기로 검색한 document 객체를 받아서, 실제 내용(page_content)만 분리해서 연결 함수

def format_docs(docs) :
    return '\n\n'.join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate([
    ('system', """당신은 '기술보증기금'에 대한 높은 이해와 해당 업무를 완벽하게 숙지하고 있는
    최고 수준의 금융/기술 컨설턴트입니다.
     제공된 [문서 정보]를 바탕으로 사용자의 질문에 상세하고 전문적으로 답변해주세요.
     
     [답변 원칙]
     1. 단답형을 피하고, 문서에 있는 핵심 내용(목적, 대상, 효과 등)을 풀어서 3~5문장 이상의 풍부한 길이로 설명할 것.
     2. 사용자가 이해하기 쉽도록 구분기호를 적극적으로 활용하여 가독성을 높일 것.
     3. 문서 정보에 없는 내용은 절대 지어내지(Hallucination)말 것.
     4. 내용이 부족하면 '제공된 문서에서는 해당 내용을 찾을 수 없습니당.' 라고 답변할 것.
     
     [문서 정보]
     {context}
     """),

     # 사용자는 human으로 role 설정
     ('human', '{question}')
])

# LCEL chain 구성
rag_chain = (
    # 딕셔너리를 쓰면 랭체인이 알아서 여러값을(병렬로 '검색기와 질의') -> prompt에 넘겨줌
    {'context': my_retriever2 | format_docs, 'question': RunnablePassthrough()}
    | prompt | chat_model | StrOutputParser()
)

answer = rag_chain.invoke('기술평가가 뭐야?')

print('최종 답변:\n', answer)

최종 답변:
 기술평가는 대상기술의 기술성, 시장성, 사업타당성 등을 분석하여 결과를 금액, 등급, 의견 등으로 표현하는 것을 말합니다. 기술평가는 기술의 가치를 평가하고 시장에서의 가치를 판단하는 중요한 역할을 합니다. 기술평가는 기술가치평가, 기술사업타당성평가, 종합기술평가 등의 유형으로 나뉘며, 기술성, 시장성, 사업성을 종합적으로 검토합니다. 이를 통해 기업이 정량화된 평가를 받아 자금조달, 투자유치, 정책자금 등 다양한 지원을 받을 수 있습니다.


### 3. Advanced RAG 사용
#### 1) Reranking (중요 문서 재판단) 사용
- Retriever가 유사도 검색을 통해 찾은 유사한 벡터를 LLM으로 전달하기 전, 벡터의 유사도를 더욱 정교하게 재평가하여 상위일부를 LLM에 전달
- Reranking을 지원하는 cohere 라이브러리 사용
- 홈페이지 회원 가입하면 key가 디폴트로 주어짐 (https://dashboard.cohere.com/api-keys)
- COHERE API KEY도 .env에 저장

In [119]:
!pip install -q langchain-cohere

In [120]:
from langchain_cohere import CohereRerank

In [121]:
load_dotenv()
COHERE_API_KEY = os.getenv('COHERE_API_KEY')

In [122]:
my_rerank = CohereRerank(
    cohere_api_key=COHERE_API_KEY,
    model='rerank-v3.5',    # cohere 지원 reranking 모델명
    top_n=3                 #  반환결과 개수
)

# 주의 : top_n의 디폴트는 3이고, 그대로 사용할 경우 my_retriever2에서 k를 5로 해도, 최종결과는 3만 반환

#### 2) Retrieve Compression(압축 검색기) 사용
- 검색된 문서의 문맥을 압축하여, 보다 집중적이고 관련성 높은 정보를 제공하는 압축 검색기
- 다른 기능 및 검색기와 통합되어 사용

In [123]:
from langchain_classic.retrievers import ContextualCompressionRetriever

In [124]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=my_rerank,         # rerank 객체
    base_retriever=my_retriever2      # retriever 객체
)

In [125]:
rag_chain_final = (
    # reranker와 문맥압축이 포함된 검색기 입력
    {'context': compression_retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt | chat_model | StrOutputParser()
)

answer = rag_chain_final.invoke('기술평가가 뭐야?')

print('최종 답변:\n', answer)

최종 답변:
 기술평가는 대상기술의 기술성, 시장성, 사업타당성 등을 분석하여 결과를 금액, 등급, 의견 등으로 표현하는 것을 말합니다. 기술평가는 기술의 가치를 평가하고 시장에서의 가치를 판단하는 중요한 요소로 활용됩니다. 이를 통해 기술의 가치를 정량화하고 기업이나 기술의 발전에 도움을 줄 수 있습니다.


- RAG 구조를 구성하게 되면 파일 데이터 자체가 OpenAI서버(다른 기업 LLM을 쓴다면 해당 기업 서버)로 가지 않으며, VectorDB도 로컬에 구성을 해뒀다면 embedding 결과는 로컬에만 머물게 됨
- 단 embedding API를 사용할 때는 데이터가 잠시 OpenAI 서버로 이동되며, OpenAI Chat모델에 질의할 때도 데이터가 서버로 이동됨
- (이를 방지하고 보안을 유지하려면 **HuggingFace와 같은 오픈소스 embedding 모델, Llama같은 오픈소스 LLM**을 사용하는 것이 좋음)
- API 호출시 사용된 데이터는 LLM 학습에 사용되지 않는 것이 기본 원칙(Zero Data Retention)이지만 대형 기업의 경우 조심해야 함

### Langsmith를 활용한 chain 관리 및 시각화
- 공식 페이지: https://smith.langchain.com/ 에서 좌측 하단의 settings 눌러서 API Key 발급받기
- 최대 3개의 워크스페이스 구성 가능, 월 10,000건 추적 가능 (이후에는 금액 부과)
- .env 파일에 4종류 변수 추가
    - LANGCHAIN_TRACING_V2=true
    - LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"
    - LANGCHAIN_API_KEY=본인의 KEY
    - LANGCHAIN_PROJECT="RAG_Project"

In [114]:
load_dotenv()

True

In [115]:
# 랭스미스 추적 끄고 싶을 때
os.environ["LANGCHAIN_TRACING_V2"] = "false"